# **Atividade Prática**
<font size=3>

- **Tema:** Raciocínio Probabilístico.
- **Prazo de entrega:** 08 de Novembro.


---

## **Enunciado:**
<font size=3>

Você precisa monitorar um **Servidor Crítico**. Existem dois problemas principais que podem derrubá-lo:
1. **Falha de Energia ($F$):** Rara, mas acontece (0.1%).
2. **Ataque Hacker ($H$):** Mais comum hoje em dia (0.5%).

Essas causas geram efeitos observáveis (sensores):
* **Servidor Cai ($S$):** Pode ser causado por **Falha de Energia** *ou* por **Ataque Hacker**. (Se nada disso acontecer, ele quase nunca cai).
* **Ar-Condicionado Desliga ($A$):** Causado exclusivamente por **Falha de Energia**.
* **Log de Segurança Alerta ($L$):** Causado exclusivamente por **Ataque Hacker**.

**Sua missão** é modelar essa rede e responder à seguinte pergunta do diretor:
> *"O servidor caiu e o ar-condicionado desligou. Qual a probabilidade de estarmos sofrendo um Ataque Hacker ao mesmo tempo?"*

<br>

Para responder este problema, siga as seguintes **etapas**.

### **Etapa 1: Modelagem e Definição das CPTs**
<font size=3>
    
Use a biblioteca `pandas` (como fizemos na **Seção 4.2** da **aula 8**) para definir as Tabelas de Probabilidade Condicional (CPTs):

* $P(F=\text{True}) = 0.001$
* $P(H=\text{True}) = 0.005$
* $P(A \mid F)$: O ar-condicionado desliga em **99%** das vezes se faltar luz. Se não faltar luz, a chance de ele desligar sozinho é mínima (**0.1%**).
* $P(L \mid H)$: O log alerta em **90%** das vezes se houver ataque hacker. Falsos alarmes ocorrem apenas **0.1%** das vezes.
* $P(S \mid F, H)$: As chances de o servidor cair dependem da causa:
    * Se houver **Falha de Energia** (com ou sem hacker): **99%** de chance de cair.
    * Se houver apenas **Ataque Hacker** (sem falha de energia): **90%** de chance de cair.
    * Se **nenhum** dos dois ocorrer: cai apenas **0.1%** das vezes.


In [1]:
import pandas as pd

In [2]:
class Factor:

    """Representa um Fator (Tabela) usando Pandas DataFrame."""

    def __init__(self, variables, data):
        self.variables = variables
        self.data = data

    def __repr__(self):
        return f"Variables: {self.variables}\n{self.data}\n"

def pointwise_product(f1, f2):

    """Operação de multiplicação de fatores."""

    # identificando variáveis comuns para fazer a fusão:
    common = list(set(f1.variables).intersection(f2.variables))

    if not common: merged = f1.data.merge(f2.data, how = 'cross') # produto Cartesiano
    else: merged = f1.data.merge(f2.data, on = common)   # inner Join

    # multiplicando as probabilidades: P_final = P1 * P2
    merged['prob'] = merged['prob_x'] * merged['prob_y']

    # limpando o DataFrame resultante:
    merged = merged.drop(columns = ['prob_x', 'prob_y'])

    new_vars = sorted(list(set(f1.variables + f2.variables)))

    return Factor(new_vars, merged)


def sum_out(var, factors):

    """operação de Eliminação (soma marginal)."""

    # separando os fatores que dependem da variável 'var':
    relevant = [f for f in factors if var in f.variables]
    irrelevant = [f for f in factors if var not in f.variables]

    if not relevant: return factors

    # multiplicando todos os relevantes: (passo de combinação)
    combined = relevant[0]
    for f in relevant[1:]:
        combined = pointwise_product(combined, f)

    # marginalização: (remove 'var' das variáveis e agrupa o resto)
    remaining_vars = [v for v in combined.variables if v != var]

    if not remaining_vars:
        total = combined.data['prob'].sum()
        new_factor = Factor([], pd.DataFrame({'prob': [total]}))
    else:
        new_data = combined.data.groupby(remaining_vars, as_index = False)['prob'].sum()
        new_factor = Factor(remaining_vars, new_data)

    return irrelevant + [new_factor]

def make_factor(var, evidencias, bn_cpts):

    """Cria fator inicial e já aplica a evidência."""

    df = bn_cpts[var].copy()

    # se alguma variável do fator está na evidência, filtramos as linhas:
    factor_vars = [c for c in df.columns if c != 'prob']
    for v in factor_vars:
        if v in evidencias:
            df = df[df[v] == evidencias[v]] # mantém só as linhas compatíveis

    return Factor(factor_vars, df)


def elimination_ask(X, e, bn_vars, bn_cpts):

    factors = [] # lista de todos os fatores iniciais

    for var in bn_vars:
        factors.append(make_factor(var, e, bn_cpts))

    # variáveis ocultas:
    hidden_vars = [v for v in bn_vars if v != X and v not in e]

    print(f"Iniciando Eliminação...\nFatores Iniciais: {len(factors)}\n")

    # elimina variáveis ocultas uma a uma:
    for var in hidden_vars:
        print(f"Eliminando (somando): {var}...")
        factors = sum_out(var, factors)

    # multiplica os fatores restantes:
    result = factors[0]
    for f in factors[1:]:
        result = pointwise_product(result, f)

    # normalização:
    total_prob = result.data['prob'].sum()
    result.data['prob'] = result.data['prob']/total_prob

    return result

In [3]:
T, F = True, False

# definindo as CPTs como DataFrames:
cpts = {'Falha de energia': pd.DataFrame({'Falha de energia': [T, F], 'prob': [0.001, 0.999]}),
        'Ataque Hacker': pd.DataFrame({'Ataque Hacker': [T, F], 'prob': [0.005, 0.995]}),

        'Servidor Cai': pd.DataFrame({
            'Falha de energia': [T,T,T,T,F,F,F,F], 
            'Ataque Hacker':    [T,T,F,F,T,T,F,F], 
            'Servidor Cai':     [T,F,T,F,T,F,T,F],
            'prob': [0.99, 0.01, 0.99, 0.01, 0.90, 0.10, 0.001, 0.999]
        }),

        'Ar-condicionado desliga': pd.DataFrame({
            'Falha de energia':         [T,T,F,F], 
            'Ar-condicionado desliga':  [T,F,T,F], 
            'prob': [0.99, 0.01, 0.001, 0.999]
        }),

        'Log de segurança alerta': pd.DataFrame({
            'Ataque Hacker':            [T,T,F,F], 
            'Log de segurança alerta':  [T,F,T,F], 
            'prob': [0.90, 0.10, 0.001, 0.999]
        })
       }

### **Etapa 2: Inferência Exata (_Eliminação de Variáveis_):**
<font size=3>
    
Utilize a função `elimination_ask` (que implementamos na **Seção 4.2**) para calcular a probabilidade exata.

* **Evidência:** `Servidor=True`, `ArCondicionado=True`.
* **Pergunta (_query_):** `Hacker`
  

In [4]:
variables_order = ['Falha de energia', 'Ataque Hacker', 'Servidor Cai', 'Ar-condicionado desliga', 'Log de segurança alerta']

evidence = {'Ar-condicionado desliga': True, 'Servidor Cai': True}
query = 'Ataque Hacker'

final_factor = elimination_ask(query, evidence, variables_order, cpts)

print(final_factor.data)

prob_ataque_hacker = final_factor.data[final_factor.data['Ataque Hacker'] == True]['prob'].values[0]
print(f"\nProbabilidade calculada de Ataque Hacker: {prob_ataque_hacker:.4f}")

Iniciando Eliminação...
Fatores Iniciais: 5

Eliminando (somando): Falha de energia...
Eliminando (somando): Log de segurança alerta...
   Ataque Hacker  Ar-condicionado desliga  Servidor Cai      prob
0           True                     True          True  0.009533
1          False                     True          True  0.990467

Probabilidade calculada de Ataque Hacker: 0.0095


De acordo com o cenário proposto, para responder ao questionamento do diretor: *O servidor caiu e o ar-condicionado desligou. Qual a probabilidade de estarmos sofrendo um Ataque Hacker ao mesmo tempo?*, basta observar o resultado e interpretá-lo. Nesse sentido, a propabilidade de existir um ataque hacker dado que o ar-condicionado desligou e o servidor caiu é de 0.0095, ou seja, 0.95%, isto é, há uma probabilidade muito baixa, próxima de 1%, de estar acontecendo um ataque hacker quando o ar-condicionado desliga e o servidor cai.

### **Etapa 3: Inferência Aproximada (_Gibbs Sampling_):**
<font size=3>
    
Agora, imagine que sua rede cresceu e a inferência exata ficou lenta. Use o **Gibbs Sampling** (**Seção 5.2**) para estimar essa mesma probabilidade.

Para isso, você precisará definir os dicionários de estrutura (`parents` e `children`) e converter as CPTs de DataFrames para Dicionários (ou adaptar a função `gibbs_ask` para ler DataFrames).

**Tarefa:**
1. Defina `parents_servidor` e `children_servidor`;
2. Defina `cpts_dict` (formato de dicionário simples, igual à **Seção 5.2**);
3. Rode o `gibbs_ask`.


In [5]:
parents_servidor = {
    'Falha de energia':[], 
    'Ataque Hacker':[],
    'Servidor Cai':['Falha de energia', 'Ataque Hacker'],
    'Ar-condicionado desliga':['Falha de energia'], 
    'Log de segurança alerta':['Ataque Hacker']
}

# precisamos identificar os filhos para o envoltório de Markov:
children_servidor = {
    'Falha de energia':['Servidor Cai', 'Ar-condicionado desliga'],
    'Ataque Hacker':['Servidor Cai', 'Log de segurança alerta'],
    'Servidor Cai':[],
    'Ar-condicionado desliga':[], 
    'Log de segurança alerta':[]
}

cpts_dict = {
    'Falha de energia': 0.001, 
    'Ataque Hacker': 0.005,
    'Servidor Cai': {(True, True): 0.99, (True, False): 0.99, (False, True): 0.90, (False, False): 0.001},
    'Ar-condicionado desliga': {True: 0.99, False: 0.001},
    'Log de segurança alerta': {True: 0.90, False: 0.001}
}

In [6]:
import random

def get_prob_local(var, current_state, bn_parents, bn_cpts):

    """Retorna P(var=True | pais_atuais)"""

    parent_names = bn_parents[var]

    if not parent_names:
        return bn_cpts[var]

    parent_vals = tuple(current_state[p] for p in parent_names)

    if len(parent_vals) == 1: parent_vals = parent_vals[0]

    return bn_cpts[var][parent_vals]


def sample_markov_blanket(var, current_state, bn_parents, bn_children, bn_cpts):

    """
    Calcula P(var | mb(var)) e sorteia um novo valor.
    Fórmula: P(x | mb) = alpha * P(x | pais) * PROD P(filhos | pais_do_filho)
    """

    # calculando o "peso" para var=True e var=False:
    weights = {}

    for val in [True, False]:
        # criando um estado temporário hipotético:
        temp_state = current_state.copy()
        temp_state[var] = val

        # 1. Termo dos Pais: P(var=val | Parents):
        p_parents = get_prob_local(var, temp_state, bn_parents, bn_cpts)
        term1 = p_parents if val else (1.0 - p_parents)

        # 2. Termo dos Filhos: PROD P(filho | Parents_do_filho):
        term2 = 1.0
        for child in bn_children[var]:
            p_child_true = get_prob_local(child, temp_state, bn_parents, bn_cpts)

            is_child_true = temp_state[child]

            term2 *= p_child_true if is_child_true else (1.0 - p_child_true)

        weights[val] = term1 * term2

    # normalização:
    total_weight = weights[True] + weights[False]
    prob_true = weights[True] / total_weight

    # sorteio:
    return random.random() < prob_true


def gibbs_ask(X, evidence, bn_vars, bn_parents, bn_children, bn_cpts, N = 2000):

    # contadores:
    counts = {True:0, False:0}

    # variáveis não-evidência:
    Z = [v for v in bn_vars if v not in evidence]

    # 1. Estado Inicial (x)
    # copia evidência e chuta valores aleatórios para Z:
    current_state = evidence.copy()
    for var in Z:
        current_state[var] = random.choice([True, False])

    # 2. Loop MCMC:
    for j in range(N):
        for var in Z:
            # atualizando o valor de Zi no estado atual amostrando do MB:
            new_val = sample_markov_blanket(var, current_state, bn_parents, bn_children, bn_cpts)
            current_state[var] = new_val

        counts[current_state[X]] += 1

    # normalização:
    total = sum(counts.values())

    return {k:v/total for k, v in counts.items()}


In [13]:
variables_order = ['Falha de energia', 'Ataque Hacker', 'Servidor Cai', 'Ar-condicionado desliga', 'Log de segurança alerta']

evidence = {'Ar-condicionado desliga': True, 'Servidor Cai': True}
query = 'Ataque Hacker'

final_factor = gibbs_ask(query, evidence, variables_order, parents_servidor, children_servidor, cpts_dict)

print(final_factor)

print(f"\nProbabilidade calculada de Ataque Hacker: {final_factor[True]:.4f}")

{True: 0.0255, False: 0.9745}

Probabilidade calculada de Ataque Hacker: 0.0255


### **Etapa 4: Análise dos resultados:**
<font size=3>
    
Compare os resultados.
1. O valor exato e o aproximado ficaram próximos?

2. **Interpretação:** Dado que o ar-condicionado desligou (o que prova a falha de energia), a probabilidade de haver *também* um ataque hacker aumentou, diminuiu ou permaneceu igual à probabilidade a priori (0.5%)?
   

Probabilidade de existir um ataque hacker dado que o ar-condicionado desligou e o servidor caiu:

* Valor exato: 0.95%
* Valor aproximado: 2.55%

Se for analisar em termos de precisão, é possível dizer que há uma diferença considerável entre o valor exato e o aproximado, sendo próxima de 1,5%, todavia se essa precisão não for tão relevante, essa diferença pode ser considerada como aceitável, pois ainda pode-se dizer que é um valor próximo do real.

Sabendo que o ar-condicionado desligou, a probabilidade de também haver um ataque hacker, para ambos os métodos utilizados, aumentou. Dessa forma, no método exato houve um pequeno aumento na probabilidade do ataque hacker, enquanto que no Gibbs Sampling acabou aumentando ainda mais, provavelmente, devido a variável sorteada que acabou aumentando as chances de um ataque hacker na situação descrita na questão. 